In [ ]:
# ============================================================
# IMPORTS
# ============================================================

from pathlib import Path

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp


# ============================================================
# PROJECT AND DATASET PATHS
# ============================================================

PROJECT_ROOT = Path(".")

DATASET_ROOT = PROJECT_ROOT

TRAIN_DIR = DATASET_ROOT / "train"
VALID_DIR = DATASET_ROOT / "valid"
TEST_DIR = DATASET_ROOT / "test"


# ============================================================
# OUTPUT PATHS
# ============================================================

OUTPUT_DIR = PROJECT_ROOT / "deeplab_results"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_MODEL_PATH = (
    OUTPUT_DIR / "best_deeplabv3plus.pt"
)

LAST_MODEL_PATH = (
    OUTPUT_DIR / "last_deeplabv3plus.pt"
)

HISTORY_PATH = (
    OUTPUT_DIR / "training_history.csv"
)


# ============================================================
# TRAINING PARAMETERS
# ============================================================

IMG_SIZE = 1024
BATCH_SIZE = 8
EPOCHS = 100

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

ENCODER = "resnet50"
ENCODER_WEIGHTS = "imagenet"

# Zero is safest for HPC Jupyter notebooks.
NUM_WORKERS = 0


# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

AMP_ENABLED = DEVICE.type == "cuda"


print("Project root:", PROJECT_ROOT.resolve())
print("Dataset root:", DATASET_ROOT.resolve())
print("Output folder:", OUTPUT_DIR.resolve())
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# READ ROBOFLOW CLASS MAPPING
# ============================================================

def load_class_map(train_directory):

    csv_path = Path(train_directory) / "_classes.csv"

    if not csv_path.is_file():
        raise FileNotFoundError(
            f"Could not find class file:\n"
            f"{csv_path.resolve()}"
        )

    dataframe = pd.read_csv(csv_path)

    dataframe.columns = [
        str(column).strip()
        for column in dataframe.columns
    ]

    required_columns = {
        "Pixel Value",
        "Class"
    }

    if not required_columns.issubset(
        dataframe.columns
    ):
        raise ValueError(
            "The _classes.csv file must contain:\n"
            "'Pixel Value' and 'Class'\n\n"
            f"Columns found: "
            f"{dataframe.columns.tolist()}"
        )

    pixel_to_class = {}

    for class_index, row in dataframe.iterrows():

        pixel_value = int(
            row["Pixel Value"]
        )

        pixel_to_class[pixel_value] = int(
            class_index
        )

    class_names = (
        dataframe["Class"]
        .astype(str)
        .str.strip()
        .tolist()
    )

    return (
        pixel_to_class,
        class_names,
        dataframe
    )


pixel_to_class, class_names, class_dataframe = (
    load_class_map(TRAIN_DIR)
)

NUM_CLASSES = len(class_names)

display(class_dataframe)

print("Class names:", class_names)
print("Number of classes:", NUM_CLASSES)
print("Pixel mapping:", pixel_to_class)

# ============================================================
# SEMANTIC SEGMENTATION DATASET
# ============================================================

class SemanticSegmentationDataset(Dataset):

    def __init__(
        self,
        split_directory,
        pixel_to_class,
        transform=None
    ):
        self.split_directory = Path(
            split_directory
        )

        self.pixel_to_class = (
            pixel_to_class
        )

        self.transform = transform

        self.samples = []

        supported_extensions = {
            ".jpg",
            ".jpeg",
            ".png"
        }

        for image_path in sorted(
            self.split_directory.iterdir()
        ):

            if (
                image_path.suffix.lower()
                not in supported_extensions
            ):
                continue

            # Exclude the semantic mask itself.
            if image_path.stem.lower().endswith(
                "_mask"
            ):
                continue

            mask_path = (
                self.split_directory
                / f"{image_path.stem}_mask.png"
            )

            if mask_path.is_file():

                self.samples.append(
                    (
                        image_path,
                        mask_path
                    )
                )

        if len(self.samples) == 0:

            raise RuntimeError(
                "No image-mask pairs were found in:\n"
                f"{self.split_directory.resolve()}\n\n"
                "Expected filenames such as:\n"
                "example.jpg\n"
                "example_mask.png"
            )

        print(
            f"{self.split_directory.name}: "
            f"{len(self.samples)} image-mask pairs"
        )

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, index):

        image_path, mask_path = (
            self.samples[index]
        )

        # -------------------------------
        # Load the RGB image
        # -------------------------------

        image = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR
        )

        if image is None:

            raise RuntimeError(
                f"Could not read image:\n"
                f"{image_path}"
            )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        # -------------------------------
        # Load semantic PNG mask
        # -------------------------------

        raw_mask = cv2.imread(
            str(mask_path),
            cv2.IMREAD_GRAYSCALE
        )

        if raw_mask is None:

            raise RuntimeError(
                f"Could not read mask:\n"
                f"{mask_path}"
            )

        # -------------------------------
        # Convert Roboflow pixel values
        # to consecutive class indices
        # -------------------------------

        mask = np.zeros(
            raw_mask.shape,
            dtype=np.int64
        )

        for pixel_value, class_index in (
            self.pixel_to_class.items()
        ):

            mask[
                raw_mask == int(pixel_value)
            ] = int(class_index)

        # -------------------------------
        # Resize, normalize and convert
        # to PyTorch tensors
        # -------------------------------

        if self.transform is not None:

            transformed = self.transform(
                image=image,
                mask=mask
            )

            image = transformed["image"]
            mask = transformed["mask"]

        return (
            image.float(),
            mask.long()
        )

# ============================================================
# PREPROCESSING ONLY — NO AUGMENTATION
# ============================================================

train_transform = A.Compose([
    A.Resize(height=IMG_SIZE,width=IMG_SIZE
    ),

    A.Normalize(
        mean= 0.485,0.456, 0.406
        ),
        std=(0.229, 0.224,0.225
        )
    ),

    ToTensorV2()
])


evaluation_transform = A.Compose([
    A.Resize(
        height=IMG_SIZE,
        width=IMG_SIZE
    ),

    A.Normalize(
        mean=(
            0.485,
            0.456,
            0.406
        ),
        std=(
            0.229,
            0.224,
            0.225
        )
    ),

    ToTensorV2()
])

# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = SemanticSegmentationDataset(
    split_directory=TRAIN_DIR,
    pixel_to_class=pixel_to_class,
    transform=train_transform
)

valid_dataset = SemanticSegmentationDataset(
    split_directory=VALID_DIR,
    pixel_to_class=pixel_to_class,
    transform=evaluation_transform
)

test_dataset = SemanticSegmentationDataset(
    split_directory=TEST_DIR,
    pixel_to_class=pixel_to_class,
    transform=evaluation_transform
)


# ============================================================
# CREATE DATA LOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=AMP_ENABLED,
    drop_last=False
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=AMP_ENABLED,
    drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=AMP_ENABLED,
    drop_last=False
)

print("Training images:", len(train_dataset))
print("Validation images:", len(valid_dataset))
print("Test images:", len(test_dataset))

print("Training batches:", len(train_loader))
print("Validation batches:", len(valid_loader))
print("Test batches:", len(test_loader))

# ============================================================
# DEEPLABV3+ MODEL
# ============================================================

model = smp.DeepLabV3Plus(
    encoder_name=ENCODER,
    encoder_weights=ENCODER_WEIGHTS,
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None
)

model = model.to(DEVICE)


# ============================================================
# LOSS FUNCTIONS
# ============================================================

dice_loss = smp.losses.DiceLoss(
    mode="multiclass",
    from_logits=True)

cross_entropy_loss = (
    nn.CrossEntropyLoss())

def combined_loss(
    predictions,
    targets):
    dice = dice_loss(
        predictions,
        targets )

    cross_entropy = (
        cross_entropy_loss(
            predictions,
            targets))

    return dice + cross_entropy


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# POLYNOMIAL LEARNING-RATE SCHEDULER
# ============================================================

def polynomial_decay(epoch):

    progress = min(
        epoch / EPOCHS,
        1.0
    )

    return (
        1.0 - progress
    ) ** 0.9


scheduler = (
    torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=polynomial_decay
    )
)


# ============================================================
# AUTOMATIC MIXED PRECISION
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)

print("DeepLabV3+ model created")
print("Encoder:", ENCODER)
print("Number of output classes:", NUM_CLASSES)
print("Model device:", next(model.parameters()).device)

# ============================================================
# METRICS
# ============================================================

class SegmentationMetrics:

    def __init__(self, num_classes):

        self.num_classes = num_classes

        self.confusion_matrix = np.zeros(
            (
                num_classes,
                num_classes
            ),
            dtype=np.int64
        )

    def update(
        self,
        outputs,
        targets
    ):

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        predictions = (
            predictions
            .detach()
            .cpu()
            .numpy()
            .reshape(-1)
        )

        targets = (
            targets
            .detach()
            .cpu()
            .numpy()
            .reshape(-1)
        )

        valid_pixels = (
            (targets >= 0)
            & (
                targets
                < self.num_classes
            )
        )

        indices = (
            self.num_classes
            * targets[valid_pixels]
            + predictions[valid_pixels]
        )

        batch_matrix = np.bincount(
            indices,
            minlength=(
                self.num_classes ** 2
            )
        ).reshape(
            self.num_classes,
            self.num_classes
        )

        self.confusion_matrix += (
            batch_matrix
        )

    def calculate(self):

        matrix = (
            self.confusion_matrix
            .astype(np.float64)
        )

        true_positive = np.diag(
            matrix
        )

        false_positive = (
            matrix.sum(axis=0)
            - true_positive
        )

        false_negative = (
            matrix.sum(axis=1)
            - true_positive
        )

        iou_denominator = (
            true_positive
            + false_positive
            + false_negative
        )

        precision_denominator = (
            true_positive
            + false_positive
        )

        recall_denominator = (
            true_positive
            + false_negative
        )

        iou = np.divide(
            true_positive,
            iou_denominator,
            out=np.full(
                true_positive.shape,
                np.nan,
                dtype=float
            ),
            where=(
                iou_denominator != 0
            )
        )

        precision = np.divide(
            true_positive,
            precision_denominator,
            out=np.full(
                true_positive.shape,
                np.nan,
                dtype=float
            ),
            where=(
                precision_denominator
                != 0
            )
        )

        recall = np.divide(
            true_positive,
            recall_denominator,
            out=np.full(
                true_positive.shape,
                np.nan,
                dtype=float
            ),
            where=(
                recall_denominator
                != 0
            )
        )

        return {
            "iou": iou,
            "precision": precision,
            "recall": recall,
            "mean_iou": np.nanmean(iou)
        }


# ============================================================
# TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader
):

    model.train()

    total_loss = 0.0

    for batch_number, (
        images,
        masks
    ) in enumerate(
        loader,
        start=1
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        masks = masks.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=AMP_ENABLED
        ):

            outputs = model(images)

            loss = combined_loss(
                outputs,
                masks
            )

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        total_loss += loss.item()

        print(
            f"\rTraining batch "
            f"{batch_number}/"
            f"{len(loader)} "
            f"| Loss: "
            f"{loss.item():.4f}",
            end=""
        )

    print()

    average_loss = (
        total_loss / len(loader)
    )

    return average_loss


# ============================================================
# VALIDATE ONE EPOCH
# ============================================================

def validate_one_epoch(
    model,
    loader
):

    model.eval()

    total_loss = 0.0

    metrics = SegmentationMetrics(
        NUM_CLASSES
    )

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            masks = masks.to(
                DEVICE,
                non_blocking=True
            )

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=AMP_ENABLED
            ):

                outputs = model(images)

                loss = combined_loss(
                    outputs,
                    masks
                )

            total_loss += loss.item()

            metrics.update(
                outputs,
                masks
            )

    average_loss = (
        total_loss / len(loader)
    )

    metric_results = (
        metrics.calculate()
    )

    return (
        average_loss,
        metric_results
    )

print("Training and validation functions ready")

# ============================================================
# TRAINING LOOP
# ============================================================

best_validation_loss = float("inf")
best_epoch = 0

training_history = []

print("=" * 70)
print("Starting DeepLabV3+ training")
print("=" * 70)


for epoch in range(
    1,
    EPOCHS + 1
):

    print()
    print(
        f"Epoch {epoch}/{EPOCHS}"
    )
    print("-" * 70)

    # ------------------------
    # Train
    # ------------------------

    train_loss = train_one_epoch(
        model,
        train_loader
    )

    # ------------------------
    # Validate
    # ------------------------

    (
        validation_loss,
        validation_metrics
    ) = validate_one_epoch(
        model,
        valid_loader
    )

    current_lr = (
        optimizer
        .param_groups[0]["lr"]
    )

    print(
        f"Training loss:   "
        f"{train_loss:.4f}"
    )

    print(
        f"Validation loss: "
        f"{validation_loss:.4f}"
    )

    print(
        f"Validation mIoU: "
        f"{validation_metrics['mean_iou']:.4f}"
    )

    print(
        f"Learning rate:   "
        f"{current_lr:.8f}"
    )

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": validation_loss,
        "mean_iou": (
            validation_metrics[
                "mean_iou"
            ]
        ),
        "learning_rate": current_lr
    }

    # ------------------------
    # Per-class metrics
    # ------------------------

    for class_index, class_name in enumerate(
        class_names
    ):

        class_iou = (
            validation_metrics[
                "iou"
            ][class_index]
        )

        class_precision = (
            validation_metrics[
                "precision"
            ][class_index]
        )

        class_recall = (
            validation_metrics[
                "recall"
            ][class_index]
        )

        print(
            f"{class_name}: "
            f"IoU={class_iou:.4f} | "
            f"Precision="
            f"{class_precision:.4f} | "
            f"Recall="
            f"{class_recall:.4f}"
        )

        safe_class_name = (
            class_name
            .lower()
            .replace(" ", "_")
            .replace("/", "_")
        )

        epoch_record[
            f"{safe_class_name}_iou"
        ] = class_iou

        epoch_record[
            f"{safe_class_name}_precision"
        ] = class_precision

        epoch_record[
            f"{safe_class_name}_recall"
        ] = class_recall

    training_history.append(
        epoch_record
    )

    # ------------------------
    # Save training history
    # ------------------------

    pd.DataFrame(
        training_history
    ).to_csv(
        HISTORY_PATH,
        index=False
    )

    # ------------------------
    # Save best model
    # ------------------------

    if (
        validation_loss
        < best_validation_loss
    ):

        best_validation_loss = (
            validation_loss
        )

        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "scheduler_state_dict":
                    scheduler.state_dict(),

                "validation_loss":
                    validation_loss,

                "mean_iou":
                    validation_metrics[
                        "mean_iou"
                    ],

                "class_names":
                    class_names,

                "pixel_to_class":
                    pixel_to_class,

                "image_size":
                    IMG_SIZE,

                "batch_size":
                    BATCH_SIZE,

                "encoder":
                    ENCODER
            },
            BEST_MODEL_PATH
        )

        print(
            "Saved new best model:",
            BEST_MODEL_PATH
        )

    # ------------------------
    # Save latest model
    # ------------------------

    torch.save(
        {
            "epoch": epoch,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scheduler_state_dict":
                scheduler.state_dict(),

            "validation_loss":
                validation_loss,

            "mean_iou":
                validation_metrics[
                    "mean_iou"
                ],

            "class_names":
                class_names,

            "pixel_to_class":
                pixel_to_class,

            "image_size":
                IMG_SIZE,

            "batch_size":
                BATCH_SIZE,

            "encoder":
                ENCODER
        },
        LAST_MODEL_PATH
    )

    scheduler.step()


print()
print("=" * 70)
print("Training completed")
print("=" * 70)

print("Best epoch:", best_epoch)

print(
    "Best validation loss:",
    best_validation_loss
)

print(
    "Best model saved to:",
    BEST_MODEL_PATH.resolve()
)

print(
    "Latest model saved to:",
    LAST_MODEL_PATH.resolve()
)

print(
    "Training history saved to:",
    HISTORY_PATH.resolve()
)

Project root: /home/user/23014319/RE7_deeplabtesting
Dataset root: /home/user/23014319/RE7_deeplabtesting
Output folder: /home/user/23014319/RE7_deeplabtesting/deeplab_results
Device: cuda
GPU: NVIDIA L4
